# Graphs in Python

Graphs model relationships between entities. They can be directed/undirected, weighted/unweighted.

## Topics Covered
- Graph Representations (Adjacency List, Adjacency Matrix)
- BFS (Queue-based)
- DFS (Stack/Recursive)
- Connected Components
- Cycle Detection
- Shortest Path - Dijkstra's Algorithm (using heap)
- Topological Sort (Kahn's Algorithm, DFS-based)
- Bipartite Graph Check

---
## 1. Graph Representations

**Adjacency List**: Dictionary mapping each vertex to its neighbors. Most common, space-efficient.

**Adjacency Matrix**: 2D array for dense graphs.

In [ ]:
# Adjacency List (most common)
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

# Directed graph with edge list -> adjacency list
def build_graph(edges):
    """Build adjacency list from edge list."""
    graph = {}
    for u, v in edges:
        if u not in graph:
            graph[u] = []
        if v not in graph:
            graph[v] = []
        graph[u].append(v)
        # For undirected: graph[v].append(u)
    return graph

# Adjacency Matrix
def build_matrix(n, edges):
    """Build n x n adjacency matrix."""
    matrix = [[0] * n for _ in range(n)]
    for u, v in edges:
        matrix[u][v] = 1
        # For undirected: matrix[v][u] = 1
    return matrix

edges = [('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','F'), ('E','F')]
g = build_graph(edges)
print("Adjacency List:")
for node in sorted(g):
    print(f"  {node}: {g[node]}")

---
## 2. BFS (Breadth-First Search)

Uses a queue. Explores nodes level by level. Finds shortest path in unweighted graphs.

In [ ]:
from collections import deque

def bfs(graph, start):
    """BFS traversal from start node."""
    visited = set()
    q = deque([start])
    visited.add(start)
    result = []
    while q:
        node = q.popleft()
        result.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                q.append(neighbor)
    return result

def bfs_shortest_path(graph, start, end):
    """Find shortest path length from start to end (unweighted)."""
    visited = {start}
    q = deque([(start, 0)])
    while q:
        node, dist = q.popleft()
        if node == end:
            return dist
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                q.append((neighbor, dist + 1))
    return -1

print(f"BFS from A: {bfs(graph, 'A')}")
print(f"Shortest path A->F: {bfs_shortest_path(graph, 'A', 'F')}")  # 2
# Time: O(V + E), Space: O(V)

---
## 3. DFS (Depth-First Search)

Uses stack (or recursion). Explores as far as possible before backtracking.

In [ ]:
def dfs_iterative(graph, start):
    """DFS traversal using explicit stack."""
    visited = set()
    stack = [start]
    result = []
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            result.append(node)
            for neighbor in graph.get(node, []):
                if neighbor not in visited:
                    stack.append(neighbor)
    return result

def dfs_recursive(graph, node, visited=None):
    """DFS traversal using recursion."""
    if visited is None:
        visited = set()
    visited.add(node)
    result = [node]
    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            result.extend(dfs_recursive(graph, neighbor, visited))
    return result

print(f"DFS iterative from A: {dfs_iterative(graph, 'A')}")
print(f"DFS recursive from A: {dfs_recursive(graph, 'A')}")
# Time: O(V + E), Space: O(V)

---
## 4. Connected Components

For undirected graphs, count the number of connected components.

In [ ]:
def count_components(graph):
    """Count number of connected components."""
    visited = set()
    count = 0
    for node in graph:
        if node not in visited:
            count += 1
            # DFS/BFS to mark entire component
            stack = [node]
            while stack:
                curr = stack.pop()
                if curr not in visited:
                    visited.add(curr)
                    for neighbor in graph.get(curr, []):
                        if neighbor not in visited:
                            stack.append(neighbor)
    return count

# Undirected graph with 3 components
g2 = {
    0: [1], 1: [0],
    2: [3, 4], 3: [2], 4: [2],
    5: []
}
print(f"Number of components: {count_components(g2)}")  # 3

---
## 5. Cycle Detection

For directed graph: use recursion stack (DFS with coloring).
For undirected: use parent tracking.

In [ ]:
def has_cycle_directed(graph):
    """Detect cycle in directed graph using DFS with colors."""
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {node: WHITE for node in graph}

    def dfs(node):
        if color[node] == GRAY:
            return True
        if color[node] == BLACK:
            return False
        color[node] = GRAY
        for neighbor in graph.get(node, []):
            if dfs(neighbor):
                return True
        color[node] = BLACK
        return False

    for node in graph:
        if color[node] == WHITE:
            if dfs(node):
                return True
    return False

def has_cycle_undirected(graph):
    """Detect cycle in undirected graph."""
    visited = set()
    def dfs(node, parent):
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                if dfs(neighbor, node):
                    return True
            elif neighbor != parent:
                return True
        return False
    for node in graph:
        if node not in visited:
            if dfs(node, None):
                return True
    return False

# Directed graph with cycle: 0->1->2->0
cyclic = {0: [1], 1: [2], 2: [0]}
acyclic = {0: [1], 1: [2], 2: []}
print(f"Directed cyclic: {has_cycle_directed(cyclic)}")  # True
print(f"Directed acyclic: {has_cycle_directed(acyclic)}")  # False

---
## 6. Topological Sort

Linear ordering of vertices in a DAG such that for every edge u->v, u comes before v.

**Kahn's Algorithm** (BFS-based): repeatedly remove nodes with indegree 0.
**DFS-based**: append after processing all descendants.

In [ ]:
from collections import deque

def topological_sort_kahn(graph):
    """Topological sort using Kahn's algorithm (BFS)."""
    indegree = {node: 0 for node in graph}
    for node in graph:
        for neighbor in graph[node]:
            indegree[neighbor] = indegree.get(neighbor, 0) + 1

    q = deque([node for node, deg in indegree.items() if deg == 0])
    result = []
    while q:
        node = q.popleft()
        result.append(node)
        for neighbor in graph.get(node, []):
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                q.append(neighbor)
    return result if len(result) == len(graph) else []  # Empty if cycle

def topological_sort_dfs(graph):
    """Topological sort using DFS."""
    visited = set()
    result = []
    def dfs(node):
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor)
        result.append(node)
    for node in graph:
        if node not in visited:
            dfs(node)
    return result[::-1]  # Reverse postorder

dag = {
    5: [2, 0],
    4: [0, 1],
    2: [3],
    3: [1],
    0: [],
    1: []
}
print(f"Topological sort (Kahn): {topological_sort_kahn(dag)}")
print(f"Topological sort (DFS): {topological_sort_dfs(dag)}")
# Time: O(V + E), Space: O(V)

---
## 7. Dijkstra's Algorithm

Finds shortest path from source to all nodes in a weighted graph (non-negative weights). Uses a min-heap.

In [ ]:
import heapq

def dijkstra(graph, start):
    """Find shortest distances from start to all nodes."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]  # (distance, node)
    while pq:
        dist, node = heapq.heappop(pq)
        if dist > distances[node]:
            continue  # Skip outdated entry
        for neighbor, weight in graph.get(node, []):
            new_dist = dist + weight
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                heapq.heappush(pq, (new_dist, neighbor))
    return distances

def dijkstra_with_path(graph, start, end):
    """Shortest path with reconstruction."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    prev = {}
    pq = [(0, start)]
    while pq:
        dist, node = heapq.heappop(pq)
        if node == end:
            break
        if dist > distances[node]:
            continue
        for neighbor, weight in graph.get(node, []):
            new_dist = dist + weight
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                prev[neighbor] = node
                heapq.heappush(pq, (new_dist, neighbor))
    # Reconstruct path
    path = []
    node = end
    while node in prev:
        path.append(node)
        node = prev[node]
    path.append(start)
    return path[::-1], distances[end]

weighted = {
    'A': [('B', 4), ('C', 2)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 8), ('E', 10)],
    'D': [('E', 2)],
    'E': []
}
print(f"Dijkstra from A: {dijkstra(weighted, 'A')}")
path, dist = dijkstra_with_path(weighted, 'A', 'E')
print(f"Shortest path A->E: {path}, distance: {dist}")
# Time: O((V + E) log V), Space: O(V)

---
## 8. Bipartite Graph Check

A graph is bipartite if we can color nodes with 2 colors such that no adjacent nodes share the same color.

In [ ]:
from collections import deque

def is_bipartite(graph):
    """Check if graph is bipartite using BFS coloring."""
    color = {}  # node -> 0 or 1
    for node in graph:
        if node not in color:
            color[node] = 0
            q = deque([node])
            while q:
                curr = q.popleft()
                for neighbor in graph.get(curr, []):
                    if neighbor not in color:
                        color[neighbor] = 1 - color[curr]
                        q.append(neighbor)
                    elif color[neighbor] == color[curr]:
                        return False
    return True

# Bipartite graph
bg = {0: [1, 3], 1: [0, 2], 2: [1, 3], 3: [0, 2]}
print(f"Bipartite: {is_bipartite(bg)}")  # True

# Triangle (not bipartite)
triangle = {0: [1], 1: [2], 2: [0]}
print(f"Triangle bipartite: {is_bipartite(triangle)}")  # False
# Time: O(V + E), Space: O(V)

---
## LeetCode Problems

### LeetCode 200 - Number of Islands

**Problem:** Count the number of islands (connected '1's) in a 2D grid.

In [ ]:
from collections import deque

def numIslands(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                q = deque([(r, c)])
                grid[r][c] = '0'  # Mark visited
                while q:
                    cr, cc = q.popleft()
                    for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == '1':
                            grid[nr][nc] = '0'
                            q.append((nr, nc))
    return count

grid = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]
print(f"Number of islands: {numIslands(grid)}")  # 3
# Time: O(m * n), Space: O(min(m,n))

### LeetCode 207 - Course Schedule (Topological Sort)

**Problem:** Can you finish all courses given prerequisites? (Cycle detection in DAG)

In [ ]:
from collections import deque

def canFinish(numCourses, prerequisites):
    graph = [[] for _ in range(numCourses)]
    indegree = [0] * numCourses
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1
    q = deque([i for i in range(numCourses) if indegree[i] == 0])
    count = 0
    while q:
        node = q.popleft()
        count += 1
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                q.append(neighbor)
    return count == numCourses

# Test
print(canFinish(2, [[1,0]]))           # True
print(canFinish(2, [[1,0],[0,1]]))     # False
# Time: O(V + E), Space: O(V + E)

### LeetCode 743 - Network Delay Time (Dijkstra)

**Problem:** Find the time it takes for all nodes to receive a signal from a source node.

In [ ]:
import heapq

def networkDelayTime(times, n, k):
    graph = [[] for _ in range(n + 1)]
    for u, v, w in times:
        graph[u].append((v, w))
    dist = [float('inf')] * (n + 1)
    dist[k] = 0
    pq = [(0, k)]
    while pq:
        d, node = heapq.heappop(pq)
        if d > dist[node]:
            continue
        for neighbor, w in graph[node]:
            nd = d + w
            if nd < dist[neighbor]:
                dist[neighbor] = nd
                heapq.heappush(pq, (nd, neighbor))
    max_delay = max(dist[1:])
    return -1 if max_delay == float('inf') else max_delay

# Test
print(networkDelayTime([[2,1,1],[2,3,1],[3,4,1]], 4, 2))  # 2
# Time: O((V + E) log V), Space: O(V + E)

### LeetCode 994 - Rotting Oranges

**Problem:** Return the minimum time for all oranges to rot. BFS from all rotten oranges simultaneously.

In [ ]:
from collections import deque

def orangesRotting(grid):
    rows, cols = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                q.append((r, c, 0))
            elif grid[r][c] == 1:
                fresh += 1
    if fresh == 0:
        return 0
    minutes = 0
    while q:
        r, c, t = q.popleft()
        minutes = t
        for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                grid[nr][nc] = 2
                fresh -= 1
                q.append((nr, nc, t + 1))
    return -1 if fresh > 0 else minutes

grid = [[2,1,1],[1,1,0],[0,1,1]]
print(f"Minutes to rot: {orangesRotting(grid)}")  # 4
# Time: O(m * n), Space: O(m * n)

---
## Summary

| Algorithm | Data Structure | Time | Use Case |
|---|---|---|---|
| BFS | Queue | O(V+E) | Shortest path (unweighted), level order |
| DFS | Stack/Recursion | O(V+E) | Connectivity, cycle detection |
| Topological Sort | Queue (Kahn's) | O(V+E) | Task scheduling |
| Dijkstra | Min-Heap | O((V+E) log V) | Shortest path (weighted) |
| Bipartite Check | BFS + colors | O(V+E) | Two-coloring |